# Veterans' preference in federal hiring — deriving the numbers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/veterans-preference-numbers/blob/main/veterans_preference_numbers.ipynb)

This notebook derives, from source, every federal-workforce figure behind the argument that
veterans' preference reaches only a minority of veterans. The OPM/EHRI numbers are queried
**live** from the public HuggingFace dataset
[`impactproject/opm-ehri-data`](https://huggingface.co/datasets/impactproject/opm-ehri-data)
with DuckDB over HTTPS — **nothing is downloaded to disk** and there are no local files to
stage. It runs top-to-bottom on Colab or any Python 3 environment.

External veteran-population figures (VA VetPop, Census ACS, DoD/CRS, MSPB) are national published
statistics, not in EHRI; each is entered with its source in the cell where it is used.

**Recurring caveat:** EHRI excludes the Postal Service and a few other bodies (intelligence
agencies, etc.), so its veteran headcount is an **undercount** of the government-wide total.
Where a figure depends on that gap, both the EHRI-as-is number and a coverage-adjusted number
are shown.

> Numbers refresh as the dataset updates, and not always slightly. Months get restated
> (134 of 258 accessions months have more than one version file, up to `v4`), and
> late-reported actions land in later files — recent years move most: CY2025 went from
> 116,076 to 126,969 hires between two reads a month apart, while CY2012 moved 0.3%.
> The point is the derivation, not a frozen snapshot.

## 0. Setup

Install DuckDB (pandas is already present on Colab). Everything else is standard library.

In [ ]:
# On Colab this installs DuckDB; locally it is a no-op if already present.
try:
    import duckdb  # noqa: F401
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])

## 1. Point DuckDB at the live dataset

The dataset stores one parquet file per month, sometimes in multiple versions
(`..._YYYYMM_vN.parquet`). We list the files straight from the HuggingFace repo API and keep
the highest version of each month. No manifest, no download — just the file index.

In [ ]:
import duckdb, json, re, urllib.request

HF   = "https://huggingface.co/datasets/impactproject/opm-ehri-data/resolve/main/"
TREE = "https://huggingface.co/api/datasets/impactproject/opm-ehri-data/tree/main?recursive=true"

# DuckDB reads parquet over https via httpfs (auto-loaded on modern builds; load explicitly to be safe).
try:
    duckdb.execute("INSTALL httpfs; LOAD httpfs;")
except Exception:
    pass

def _all_parquet():
    """Every parquet path in the dataset, from the HuggingFace repo tree API."""
    req = urllib.request.Request(TREE, headers={"User-Agent": "veterans-preference-numbers"})
    tree = json.load(urllib.request.urlopen(req, timeout=120))
    return [d["path"] for d in tree if d.get("type") == "file" and d["path"].endswith(".parquet")]

ALL_FILES = _all_parquet()
print(f"{len(ALL_FILES):,} parquet files in the dataset")

def latest_files(prefix, min_ym="000000"):
    """{YYYYMM: url} keeping the highest version per month for the given table prefix."""
    best = {}
    for k in ALL_FILES:
        m = re.search(rf"{prefix}_(\d{{6}})_v(\d+)\.parquet", k)
        if m and m.group(1) >= min_ym:
            ym, v = m.group(1), int(m.group(2))
            if ym not in best or v > best[ym][0]:
                best[ym] = (v, k)
    return {ym: HF + p for ym, (_, p) in best.items()}

def q1(sql):
    return duckdb.execute(sql).fetchone()[0]

emp = latest_files("employment")
EMP_URL = emp[max(emp)]          # latest point-in-time workforce snapshot
print("latest employment snapshot:", max(emp))

# Collect (claim, computed, note) rows for a summary table at the end.
checks = []
def record(claim, computed, note=""):
    checks.append((claim, computed, note))
    tail = f"   [{note}]" if note else ""
    print(f"{claim}\n   = {computed}{tail}\n")

## 2. External veteran-population inputs (not from EHRI)

The denominators. These are published national statistics, entered with their sources.

In [ ]:
# Total living veterans -- VA VetPop2023 (data.va.gov)
TOTAL_VETERANS = 18_300_000
# Share of veterans 65+ -- Census "Aging Veterans" (ACS 2021): ~8.1M of ~16.5M ~ 49%
SHARE_65_PLUS = 0.49
WORKING_AGE_VETERANS = round(TOTAL_VETERANS * (1 - SHARE_65_PLUS))
# New veterans / year -- DoD "over 200,000 transition each year" (CRS R48114 / military.com)
NEW_VETERANS_PER_YEAR = 200_000
# National veteran bachelor's+ rate -- Census ACS 2022: 31.6%
NAT_VET_BACHELORS = 0.316

record("all living veterans (VA VetPop2023)", f"{TOTAL_VETERANS:,}")
record("working-age (under 65) veterans", f"{WORKING_AGE_VETERANS:,}",
       "18.3M x (1 - 0.49 aged 65+)")
record("new veterans per year (DoD/CRS)", f"{NEW_VETERANS_PER_YEAR:,}")

## 3. The stock: veterans in a federal job right now

How many veterans currently hold a federal job, and what share of the workforce and of the
veteran population that is.

In [ ]:
vet = q1(f"SELECT SUM(CAST(count AS BIGINT)) FROM read_parquet('{EMP_URL}') WHERE veteran_indicator='Y'")
tot = q1(f"SELECT SUM(CAST(count AS BIGINT)) FROM read_parquet('{EMP_URL}')")
record("veterans holding federal jobs (EHRI)", f"{vet:,}")
record("veteran share of federal workforce", f"{100*vet/tot:.1f}%")

# As a share of the veteran population -- shown both EHRI-as-is and coverage-adjusted,
# because EHRI omits USPS (which DOES apply veterans' preference) + a few agencies.
UPLIFT  = 1.15
vet_adj = round(vet * UPLIFT)
print("--- as % of working-age veterans (9.3M) ---")
print(f"   EHRI as-is         : {100*vet/WORKING_AGE_VETERANS:.1f}%")
print(f"   coverage-adjusted  : {100*vet_adj/WORKING_AGE_VETERANS:.1f}%")
print("--- as % of all veterans (18.3M) ---")
print(f"   EHRI as-is         : {100*vet/TOTAL_VETERANS:.1f}%")
print(f"   coverage-adjusted  : {100*vet_adj/TOTAL_VETERANS:.1f}%")
record("share of 9.3M working-age vets", f"{100*vet/WORKING_AGE_VETERANS:.1f}% as-is / {100*vet_adj/WORKING_AGE_VETERANS:.1f}% coverage-adj")
record("share of all 18.3M vets", f"{100*vet/TOTAL_VETERANS:.1f}% as-is / {100*vet_adj/TOTAL_VETERANS:.1f}% coverage-adj")

**Note on which number to quote.** The headline count (~560k) and workforce share (~28%) are
EHRI-as-is. But "about 7% of working-age veterans / ~3.5% of all veterans" only holds with the
coverage-adjusted ~644k; the as-is 560k gives ~6% / ~3%. Quote a consistent pair.

## 4. The flow, 2010–2024: annual veteran new hires

Real annual veteran **new hires** — inter-agency transfers excluded — in one DuckDB query over
all monthly accession files. `veteran_indicator` exists from 2010 on.

Rows are counted by their **effective month, wherever they were filed**. A monthly file also
carries actions effective in earlier months; those are new records rather than repeats of ones
already filed (checked on effective months 202403, 202412 and 202506 — the rows appearing in
later files share zero keys with the month's own file across all 67 dimension columns). Keeping
only rows whose effective month equals their file's month would discard them, which understates
the most recent years, where the late filings have not all landed yet.

In [ ]:
acc = latest_files("accessions", min_ym="201001")  # veteran_indicator exists from 2010
urls = "[" + ",".join(f"'{u}'" for u in acc.values()) + "]"
q = f"""
WITH src AS (
  SELECT personnel_action_effective_date_yyyymm eff,
         accession_category cat, veteran_indicator vet, CAST(count AS BIGINT) c
  FROM read_parquet({urls}, union_by_name=true)
)
SELECT substr(eff,1,4) AS yr,
       SUM(c) FILTER (WHERE cat NOT LIKE 'TRANSFER%') new_hires,
       SUM(c) FILTER (WHERE cat NOT LIKE 'TRANSFER%' AND vet='Y') vet_hires
FROM src WHERE eff >= '201001'   -- 2010+ files still carry a few older effective dates
GROUP BY 1 ORDER BY 1
"""
hist = duckdb.execute(q).df()
hist["vet_share"] = (100*hist.vet_hires/hist.new_hires).round(1)
print(hist.to_string(index=False))

In [ ]:
h = hist.set_index("yr")
full = h.loc["2010":"2024", "vet_hires"]
record("veteran hires/yr range, 2010-2024",
       f"min {int(full.min()):,} ({full.idxmin()}), max {int(full.max()):,} ({full.idxmax()})")

# Veteran share of new hires: early 2010s vs recent
early = 100*h.loc["2010":"2015","vet_hires"].sum()/h.loc["2010":"2015","new_hires"].sum()
late  = 100*h.loc["2023":"2024","vet_hires"].sum()/h.loc["2023":"2024","new_hires"].sum()
record("vet share of new hires, 2010-2015", f"{early:.1f}%")
record("vet share of new hires, 2023-24",  f"{late:.1f}%")

## 5. 2024 vs 2025: the hiring-freeze collapse

Veteran new hires as a share of the 9.3M working-age veterans, and how far that rate fell
between 2024 and the 2025 freeze.

In [ ]:
for yr in ("2024", "2025"):
    n = int(h.loc[yr, "vet_hires"])
    pct = 100*n/WORKING_AGE_VETERANS
    one_in = round(1/(n/WORKING_AGE_VETERANS))
    record(f"{yr} veteran new hires", f"{n:,}")
    record(f"{yr} as % of working-age vets", f"{pct:.2f}%  (about one in {one_in})")

drop = 1 - (h.loc["2025","vet_hires"]/h.loc["2024","vet_hires"])
record("fall in veteran hire count, 2024->2025", f"{100*drop:.0f}%")

## 6. Education: federally-employed veterans vs veterans overall

Share with a bachelor's degree or higher, among employees with known education
(`NO DATA REPORTED` dropped). Compared to the national veteran rate from Census ACS.

In [ ]:
ba = "('BACHELORS DEGREE','MASTERS OR PROFESSIONAL DEGREE','DOCTORATE DEGREE')"
row = duckdb.execute(f"""
  SELECT SUM(CAST(count AS BIGINT)) tot,
         SUM(CAST(count AS BIGINT)) FILTER (WHERE education_level_bracket='NO DATA REPORTED') nd,
         SUM(CAST(count AS BIGINT)) FILTER (WHERE education_level_bracket IN {ba}) baplus
  FROM read_parquet('{EMP_URL}') WHERE veteran_indicator='Y'""").fetchone()
fed_ba = row[2]/(row[0]-row[1])
record("federally-employed vets with bachelor's+", f"{100*fed_ba:.1f}%")
record("veterans overall with bachelor's+", f"{100*NAT_VET_BACHELORS:.1f}%", "Census ACS 2022 (external)")

## 7. Where preference bites: competitive service vs excepted service

Preference applies in **competitive examining**, not under direct-hire authority. MSPB's
*Direct-Hire Authority Under 5 U.S.C. § 3304* (2021) found that in FY2014–2018 veterans were
**50% of competitive-examining hires but only 22% of §3304 direct-hire hires** (preference
waived) — that gap is preference's fingerprint (external to EHRI).

EHRI has no hiring-authority field, so it can't split competitive examining from direct hire.
What it *can* show — as corroboration — is the veteran share within the **competitive service**
vs the **excepted service** over time.

In [ ]:
q = f"""
WITH src AS (
  SELECT substr(regexp_extract(filename,'accessions_(\\d{{6}})_v',1),1,4) yr,
         personnel_action_effective_date_yyyymm eff,
         regexp_extract(filename,'accessions_(\\d{{6}})_v',1) snap,
         accession_category cat, veteran_indicator vet, CAST(count AS BIGINT) c
  FROM read_parquet({urls}, filename=true, union_by_name=true)
), n AS (SELECT * FROM src WHERE eff=snap AND cat NOT LIKE 'TRANSFER%')
SELECT yr,
  ROUND(100.0*SUM(c) FILTER (WHERE cat LIKE '%COMPETITIVE%' AND vet='Y')
        /NULLIF(SUM(c) FILTER (WHERE cat LIKE '%COMPETITIVE%'),0),1) comp_vet_pct,
  ROUND(100.0*SUM(c) FILTER (WHERE cat LIKE '%EXCEPTED%' AND vet='Y')
        /NULLIF(SUM(c) FILTER (WHERE cat LIKE '%EXCEPTED%'),0),1) excepted_vet_pct
FROM n GROUP BY yr ORDER BY yr
"""
chan = duckdb.execute(q).df()
print("veteran share within each track, by year (EHRI):")
print(chan.to_string(index=False))
print()
record("MSPB: vet share of competitive-examining hires (FY14-18)", "50%", "external -- MSPB DHA brief")
record("MSPB: vet share of direct-hire hires (preference waived)", "22%", "external -- MSPB DHA brief")
c = chan.set_index("yr")["comp_vet_pct"]
record("EHRI competitive-service vet share, 2011 -> 2024 (corroborates)",
       f"{c.loc['2011']:.0f}% (2011) -> {c.loc['2024']:.0f}% (2024)")

## 8. The "ever" estimate: what share of veterans ever hold a federal job

The ceiling on preference's reach, under the deliberate over-assumption that *every* veteran in
a federal job got there via preference.

First we **measure** the first-time share from length-of-service at hire, rather than assuming
it: a veteran hired with under 1 year of service is an unambiguous first-timer (no prior federal
time, no military credit showing) — a hard floor. Non-veterans are the returner control (no
military-credit confound). Then two independent methods — flow, and stock × turnover — settle
the number.

In [ ]:
# First-time share, measured from length-of-service at hire (all years 2010+, transfers excluded)
q = f"""
WITH src AS (
  SELECT TRY_CAST(length_of_service_years AS DOUBLE) los, veteran_indicator v, CAST(count AS BIGINT) c,
         personnel_action_effective_date_yyyymm eff,
         regexp_extract(filename,'accessions_(\\d{{6}})_v',1) snap, accession_category cat
  FROM read_parquet({urls}, filename=true, union_by_name=true)
)
SELECT v, SUM(c) FILTER (WHERE los IS NOT NULL) known, SUM(c) FILTER (WHERE los<1) lt1
FROM src WHERE eff=snap AND cat NOT LIKE 'TRANSFER%' GROUP BY v
"""
r = {row[0]:(row[1],row[2]) for row in duckdb.execute(q).fetchall()}
ft_floor   = r["Y"][1]/r["Y"][0]        # veteran LOS<1: hard first-time floor
nonvet_ret = 1 - r["N"][1]/r["N"][0]    # non-vet returner rate (control)
ft_high    = 1 - nonvet_ret             # first-time if vet returners == control
ft_central = (ft_floor + ft_high)/2
print(f"veteran first-time floor (LOS<1) : {ft_floor:.0%}")
print(f"non-vet returner rate (control)  : {nonvet_ret:.0%}")
print(f"=> first-time share: {ft_floor:.0%} floor .. {ft_high:.0%} high, central {ft_central:.0%}")

In [ ]:
# Settle the "ever" number two independent ways.
COVER = 1.15                                    # USPS (has preference) + omitted agencies
hires = round((h.loc["2022","vet_hires"]+h.loc["2023","vet_hires"]+h.loc["2024","vet_hires"])/3)
flow  = lambda ft: hires*ft*COVER/NEW_VETERANS_PER_YEAR

FED_VET_PEW = 713_000                            # Pew/OPM Sep-2024, incl. postal (pre-2025 cuts)
stock_share = FED_VET_PEW/WORKING_AGE_VETERANS
stock = lambda L: stock_share*30/L              # working-age ~30 yrs / avg federal tenure L

flow_c, stock_c = flow(ft_central), stock(11)
lo = min(flow(ft_floor), stock(13)); hi = max(flow(ft_high), stock(9))
print(f"recent-normal veteran hires/yr : {hires:,}  (2022-24 mean)")
print(f"FLOW  method : {flow(ft_floor):.0%}..{flow(ft_high):.0%}  central {flow_c:.0%}")
print(f"STOCK method : {stock(13):.0%}..{stock(9):.0%}  central {stock_c:.0%}")
cen = (flow_c+stock_c)/2
record("share of veterans who EVER hold a federal job",
       f"~{cen:.0%} central (1 in {round(1/cen)}); range {lo:.0%}-{hi:.0%}",
       "CEILING: credits preference with every veteran hire")

## 9. Summary

Everything derived above, in one table.

In [ ]:
import pandas as pd
df = pd.DataFrame(checks, columns=["figure", "computed", "note"])
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
print(df.to_string(index=False))